In [1]:
import os
import json
from colorama import init, Fore
from typing import List

# Packages for Data Loader, Preprocessing ,Embeddings, Vector DB store
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Packages for LLM model, Prompt + RAG chain
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


In [2]:
with open("../utils/credentials.json") as f:
    credentials = json.load(f)

os.environ["GROQ_API_KEY"] = credentials['Groq']['API_Key']

In [3]:
def build_or_load_vectorstore(pdf_path: str, faiss_dir: str = "faiss_index") -> FAISS:
    if os.path.isdir(faiss_dir):
        # Load existing FAISS index
        vs = FAISS.load_local(faiss_dir, embeddings=HuggingFaceEmbeddings())
        return vs
    
    loader = PyPDFLoader(pdf_path)

    # Dtype: List[Document]
    docs = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size = 1000,
        chunk_overlap = 150,
        add_start_index = True,
        separators = ["\n\n", "\n", " ", ""]
    )
    chunks = splitter.split_documents(docs)

    embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-mpnet-base-v2")

    vs = FAISS.from_documents(chunks,embeddings)
    vs.save_local(faiss_dir)

    return vs

In [9]:
def build_rag_chain(vs: FAISS):
    retriever = vs.as_retriever(search_type = "similarity", search_kwgs = {"k":4})

    llm = ChatGroq(
        model="llama-3.1-8b-instant",
        temperature=0.7,
        max_tokens=None,
    )

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system",
             "You are a helpful assistant for question answering over a provided document. "
             "Use ONLY the given context to answer. If the answer isn't in the context, say you don't know."),
            ("human",
             "Question: {question}\n\nContext:\n{context}\n\nAnswer:")
        ]
    )

    def format_docs(docs: List[Document]) -> str:
        # include source metadata for traceability
        return "\n\n---\n\n".join(
            f"[p{d.metadata.get('page', 'NA')}] {d.page_content.strip()}" for d in docs
        )

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return rag_chain

### Creating a vector store

In [6]:
vectorstore = build_or_load_vectorstore(
    pdf_path="../docs/attention_transformer_research_paper.pdf",
    faiss_dir="../database/vector_db/faiss_index"
)

/home/surya/anaconda3/envs/gen_ai/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Building a RAG agent

In [10]:
chain = build_rag_chain(vs=vectorstore)

In [11]:
while True:
    query = input("Enter your question: ")
    if query.lower() == "quit":
        break
    
    print(Fore.BLUE + f"{query}")
    answer = chain.invoke(input=query)
    print(Fore.GREEN + f"Answer: \n {answer}\n\n")

what is the research paper about?
Answer: 
 Unfortunately, the provided context doesn't explicitly state what the research paper is about. However, it appears to be a list of references related to various papers in the fields of neural networks, computer vision, and natural language processing.


can you provide me the gist of the article?
Answer: 
 The provided document appears to be a research paper or a thesis, covering various topics in the field of artificial intelligence and machine learning. However, the "gist" of the article is not explicitly stated in the given context.

From the provided text, we can infer that the article discusses various techniques and methods in natural language processing and machine translation, including attention mechanisms, neural networks, and optimization methods. The document also references several research papers and studies in the field.

To provide a more accurate answer, I would need more information about the article's content, such as its t